# 24. 모델 학습 및 하이퍼파라미터 최적화 (Self-Contained Pipeline)

이 노트북은 README의 6장에 기술된 **UnderBagging Ensemble** 방식과 **단일 하이퍼파라미터 튜닝** 파이프라인을 독립적으로 실행할 수 있도록 통합 구성되었습니다.
이 노트북은 외부 스크립트(`src/train_core.py` 또는 `config/train_config.py`)에 의존하지 않고 자체적으로 구동됩니다.

### 실행 단계
1. **환경 및 설정**: 로컬 경로, 탐색 공간 및 하이퍼파라미터 범위 구성
2. **학습 엔진 구성**: SubsetTrainer, UnderbaggingEnsemble 및 각종 유틸리티 클래스 정의
3. **데이터 로드**: Full Train (10 Subsets), Full Validation, Sampled Validation 로드
4. **[Optuna Tuning] 하이퍼파라미터 탐색**: 넓은 탐색 공간, 가지치기(Pruning) 활성화, `n_estimators` 포함
5. **[Reranking] 최종 파라미터 선정**: Sampled 검증셋의 편향 리스크를 줄이기 위해 상위 후보들을 Full Validation으로 재평가
6. **최종 모델 저장**: 산출된 최적 파라미터 및 앙상블 모델을 저장

## 1. 라이브러리 임포트 및 독립 설정 구역

In [ ]:
import sys, os, warnings, itertools, joblib, json, uuid, shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, confusion_matrix

warnings.filterwarnings("ignore")

# ── 한글 폰트 설정 ──────────────────────────────────────────
def _set_korean_font():
    candidates = ["Malgun Gothic", "NanumGothic", "AppleGothic", "DejaVu Sans"]
    for name in candidates:
        if any(name.lower() in f.name.lower() for f in fm.fontManager.ttflist):
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False

_set_korean_font()

# ── [독립 설정 클래스] ────────────────────────────────────────
class LocalConfig:
    # 1) 데이터 및 모델 저장 경로 설정
    DATA_ROOT = "../data2"      # 데이터 루트 폴더 (data 또는 data2)
    SUBSET_DIR = f"{DATA_ROOT}/06_subset_generation/seed_42"
    VAL_TUNE_PATH = f"{DATA_ROOT}/03_splitting/val_tune.parquet"
    VAL_TUNE_SAMPLED_PATH = f"{DATA_ROOT}/06_subset_generation/seed_42/val_sampled.parquet"
    MODEL_SAVE_DIR = "./models2/06d_optuna_tuning/seed_42"

    # 2) 학습 타겟 변수 및 평가 임계값 목록
    TARGET_COL = "failure"
    SEED = 42
    EVAL_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]

    # 3) LightGBM 기본 파라미터 (고정 항목들)
    LGBM_PARAMS = {
        "bagging_freq": 1,
        "verbosity": -1,
        "device": "cpu",
        "random_state": 42,
        "max_bin": 63,
    }

    # 4) Optuna 탐색 바운드 정의
    OPTUNA_BOUNDS = {
        "learning_rate": [0.005, 0.2],
        "max_depth": [3, 8],
        "num_leaves": [15, 255],
        "min_child_samples": [5, 100],
        "feature_fraction": [0.5, 1.0],
        "bagging_fraction": [0.5, 1.0],
        "n_estimators": [50, 300],
        "lambda_l1": [1e-8, 10.0],
        "lambda_l2": [1e-8, 10.0],
    }

    # 5) Optuna 튜닝 설정
    OPTUNA_TRIALS = 30
    OPTUNA_TIMEOUT = 3600
    OPTUNA_DB_PATH = "optuna_study.db"
    OPTUNA_STUDY_NAME = "hdd_failure_prediction_seed_42"

print("✅ 독립 실행 환경 및 경로 구성 완료")

## 2. 언더배깅 앙상블 핵심 엔진 코드

In [ ]:
@dataclass
class SubsetResult:
    """서브셋 하나의 학습 결과."""
    subset_id: int
    model: LGBMClassifier
    val_prauc: float
    n_train_pos: int
    n_train_neg: int

@dataclass
class EnsembleResult:
    """앙상블 최종 결과."""
    subset_results: list[SubsetResult]
    val_tune_prauc: float
    val_tune_probs: np.ndarray
    val_tune_y_true: np.ndarray
    models: list[LGBMClassifier] = field(default_factory=list)

    def __post_init__(self):
        self.models = [r.model for r in self.subset_results]

class SubsetTrainer:
    """단일 서브셋 모델 학습기."""
    def __init__(self, lgbm_params: dict, target_col: str = "failure"):
        self.lgbm_params = lgbm_params
        self.target_col = target_col
        self._meta = {"serial_number", "date", "days_to_failure", target_col}

    def _get_features(self, df: pd.DataFrame) -> list[str]:
        return [c for c in df.columns if c not in self._meta]

    def train(
        self,
        subset_id: int,
        df_subset: pd.DataFrame,
        feature_cols: Optional[list[str]] = None,
    ) -> SubsetResult:
        feats = feature_cols or self._get_features(df_subset)

        X_tr = df_subset[feats]
        y_tr = df_subset[self.target_col]

        model = LGBMClassifier(**self.lgbm_params)
        model.fit(X_tr, y_tr)

        return SubsetResult(
            subset_id=subset_id,
            model=model,
            val_prauc=0.0,
            n_train_pos=int(y_tr.sum()),
            n_train_neg=int((y_tr == 0).sum()),
        )

class UnderbaggingEnsemble:
    """비대칭 언더배깅 앙상블 (학습 후 일괄 검증 방식)."""
    def __init__(self, trainer: SubsetTrainer):
        self.trainer = trainer
        self._result: Optional[EnsembleResult] = None

    def fit(
        self,
        df_train_list: list[pd.DataFrame],
        df_val_tune: pd.DataFrame,
        feature_cols: Optional[list[str]] = None,
        target_col: str = "failure",
        trial: Optional[optuna.Trial] = None,
    ) -> EnsembleResult:
        subsets = df_train_list
        feats = feature_cols or [
            c for c in df_val_tune.columns
            if c not in {"serial_number", "date", "days_to_failure", target_col}
        ]

        # 특성 스키마 유효성 검증
        missing_val = set(feats) - set(df_val_tune.columns)
        if missing_val:
            raise ValueError(f"❌ [Error] 검증 데이터에 다음 특성이 누락되었습니다: {missing_val}")
            
        for i, sub in enumerate(subsets):
            missing_sub = set(feats) - set(sub.columns)
            if missing_sub:
                raise ValueError(f"❌ [Error] 훈련 서브셋 {i}에 다음 특성이 누락되었습니다: {missing_sub}")

        X_val = df_val_tune[feats].astype(np.float32)
        y_val = df_val_tune[target_col]

        subset_results: list[SubsetResult] = []
        probs_list = []
        is_optuna = trial is not None
        
        for i, sub in enumerate(subsets):
            if is_optuna:
                msg = f"  🏋️  Trial {trial.number} - Subset {i+1}/{len(subsets)} 학습 중..."
            else:
                msg = f"  🏋️  Subset {i+1}/{len(subsets)} 학습 중..."
            print(f"\r{msg.ljust(60)}", end="", flush=True)
            res = self.trainer.train(
                subset_id=i,
                df_subset=sub,
                feature_cols=feature_cols,
            )
            subset_results.append(res)
            
            p = res.model.predict_proba(X_val)[:, 1]
            probs_list.append(p)
            res.val_prauc = average_precision_score(y_val, p)
            
            if trial is not None:
                cur_probs = np.mean(probs_list, axis=0)
                cur_score = average_precision_score(y_val, cur_probs)
                trial.report(cur_score, step=i)
                if trial.should_prune():
                    print(f"\r  🚫  [Pruned] Trial {trial.number} pruned at step {i} (score: {cur_score:.5f})".ljust(60))
                    raise optuna.TrialPruned()

        if is_optuna:
            print("\r" + " " * 60 + "\r", end="", flush=True)
        else:
            print(f"\r✅  {len(subsets)}개 모델 학습 및 평가 완료.".ljust(60))

        probs = np.mean(probs_list, axis=0)
        ensemble_prauc = average_precision_score(y_val, probs)

        self._result = EnsembleResult(
            subset_results=subset_results,
            val_tune_prauc=ensemble_prauc,
            val_tune_probs=probs,
            val_tune_y_true=y_val.values if hasattr(y_val, "values") else y_val,
        )
        return self._result

## 3. Optuna 목적함수 & Rerank 엔진 및 최적화 실행 함수

In [ ]:
def make_optuna_objective(
    df_train: list[pd.DataFrame],
    df_val_tune: pd.DataFrame,
    feature_cols: list[str],
    target_col: str = "failure",
    device: str = "cpu",
    bounds: dict = None,
    save_model_dir: str = None,
):
    if bounds is None:
        raise ValueError("❌ 'bounds' (탐색 범위)가 지정되지 않았습니다.")

    def objective(trial):
        max_depth   = trial.suggest_int("max_depth", *bounds["max_depth"])
        max_leaves  = min(2 ** max_depth, bounds["num_leaves"][1])
        min_leaves  = min(bounds["num_leaves"][0], max_leaves)
        num_leaves  = trial.suggest_int("num_leaves", min_leaves, max_leaves)
        
        n_estimators = trial.suggest_int("n_estimators", *bounds["n_estimators"])
        scale_pos_weight = trial.suggest_float("scale_pos_weight", *bounds["scale_pos_weight"]) if "scale_pos_weight" in bounds else 1.0

        params = {
            "learning_rate":     trial.suggest_float("learning_rate", *bounds["learning_rate"], log=True),
            "max_depth":         max_depth,
            "num_leaves":        num_leaves,
            "min_child_samples": trial.suggest_int("min_child_samples", *bounds["min_child_samples"]),
            "feature_fraction":  trial.suggest_float("feature_fraction", *bounds["feature_fraction"]),
            "bagging_fraction":  trial.suggest_float("bagging_fraction", *bounds["bagging_fraction"]),
            "bagging_freq":      1,
            "lambda_l1":         trial.suggest_float("lambda_l1", *bounds["lambda_l1"], log=True),
            "lambda_l2":         trial.suggest_float("lambda_l2", *bounds["lambda_l2"], log=True),
            "n_estimators":      n_estimators,
            "scale_pos_weight":  scale_pos_weight,
            "verbosity":         -1,
            "device":            device,
            "random_state":      42,
            "max_bin":           63,
        }

        trainer = SubsetTrainer(lgbm_params=params, target_col=target_col)
        ens     = UnderbaggingEnsemble(trainer=trainer)
        result  = ens.fit(df_train, df_val_tune, feature_cols=feature_cols, target_col=target_col, trial=trial)
        
        if save_model_dir is not None:
            trial_id_str = f"trial_{trial.number}_{uuid.uuid4().hex[:8]}"
            trial_dir = Path(save_model_dir) / trial_id_str
            trial_dir.mkdir(parents=True, exist_ok=True)
            for i, model in enumerate(result.models):
                joblib.dump(model, trial_dir / f"model_{i}.pkl")
            trial.set_user_attr("model_dir", trial_id_str)
                
        return result.val_tune_prauc

    return objective

def run_training(
    cfg,
    feature_cols: Optional[list[str]] = None,
    *,
    run_optuna: bool = False,
    optuna_trials: Optional[int] = None,
    optuna_timeout: Optional[int] = None,
    optuna_rerank_delta: float = 0.005,
    optuna_rerank_cap: int = 5,
    optuna_rerank_ids: Optional[list[int]] = None,
    interactive_rerank: bool = False,
    cleanup_optuna_temp: bool = True,
) -> dict:
    # 1. 파일 검증 및 로드
    subset_dir = Path(cfg.SUBSET_DIR)
    subset_files = sorted(list(subset_dir.glob("subset_*.parquet"))) if subset_dir.exists() else []
    if not subset_files:
        raise FileNotFoundError(f"❌ [Error] 사전 분할 데이터가 {subset_dir}에 없습니다.")

    df_train = [pd.read_parquet(f) for f in subset_files]
    # ⚠️ 안전장치: 고장 당일(D-DAY) 행 제외 (이미 제외되었을 수 있으나 이중 보장)
    df_train_cleaned = []
    for sub in df_train:
        sub_failed_serials = sub[sub[cfg.TARGET_COL] == 1]['serial_number'].unique()
        if len(sub_failed_serials) > 0:
            df_failed_sub = sub[sub['serial_number'].isin(sub_failed_serials)]
            max_dates = df_failed_sub.groupby('serial_number')['date'].max().reset_index()
            max_dates['is_dday'] = True
            sub_cleaned = sub.merge(max_dates, on=['serial_number', 'date'], how='left')
            sub_cleaned = sub_cleaned[sub_cleaned['is_dday'].isna()].drop(columns=['is_dday'])
            df_train_cleaned.append(sub_cleaned)
        else:
            df_train_cleaned.append(sub.copy())
    df_train = df_train_cleaned
    
    val_tune_path = Path(cfg.VAL_TUNE_PATH)
    if not val_tune_path.exists():
        raise FileNotFoundError(f"❌ [Error] 원본 검증셋(VAL_TUNE_PATH)이 존재하지 않습니다.")
    df_val_tune_full = pd.read_parquet(val_tune_path)

    print(f"  [Debug] Train Subsets: {len(df_train)} files")
    print(f"  [Debug] Val Tune (Full) Rows: {len(df_val_tune_full):,}")
    
    sampled_path = Path(cfg.VAL_TUNE_SAMPLED_PATH)
    if run_optuna:
        if not sampled_path.exists():
            raise FileNotFoundError(f"❌ [Error] Optuna 모드에서는 샘플링된 검증셋({sampled_path})이 필수입니다.")
        df_val_optuna = pd.read_parquet(sampled_path)
        print(f"  [Debug] Val Tune (Sampled for Optuna) loaded: {len(df_val_optuna):,} rows")
        is_val_sampled = True
    else:
        if sampled_path.exists():
            df_val_optuna = pd.read_parquet(sampled_path)
            print(f"  [Debug] Val Tune (Sampled for Single Run) loaded: {len(df_val_optuna):,} rows")
            is_val_sampled = True
        else:
            df_val_optuna = df_val_tune_full
            print(f"  [Debug] Val Tune (Full for Single Run) Rows: {len(df_val_optuna):,}")
            is_val_sampled = False

    feats = feature_cols
    _device = cfg.LGBM_PARAMS.get("device", "cpu")

    # 2. Optuna 튜닝 시작
    best_params = cfg.LGBM_PARAMS.copy()
    if run_optuna:
        db_path = cfg.OPTUNA_DB_PATH
        base_study_name = cfg.OPTUNA_STUDY_NAME
        storage_url = f"sqlite:///{db_path}"

        trials = optuna_trials if optuna_trials is not None else cfg.OPTUNA_TRIALS
        print(f"\n  [Optuna Tuning] 하이퍼파라미터 탐색 시작 (설정 목표: {trials}회, 가지치기 활성화)")
        
        optuna_temp_dir = Path(cfg.MODEL_SAVE_DIR).parent / "optuna_temp"
        
        study = optuna.create_study(
            study_name=base_study_name,
            storage=storage_url,
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=cfg.SEED),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
            load_if_exists=True,
        )
        
        total_trials = len(study.trials)
        if total_trials < trials:
            obj = make_optuna_objective(
                df_train, df_val_optuna, feats, cfg.TARGET_COL, _device,
                bounds=cfg.OPTUNA_BOUNDS, save_model_dir=str(optuna_temp_dir)
            )
            study.optimize(obj, n_trials=trials - total_trials, timeout=optuna_timeout)
        
        valid_trials = [t for t in study.trials if t.number < trials and t.state == optuna.trial.TrialState.COMPLETE]
        if not valid_trials:
            raise ValueError(f"완료된 트라이얼이 없습니다. (목표 횟수: {trials})")
            
        best_trial = max(valid_trials, key=lambda t: t.value)
        print(f"  [Optuna Tuning] 완료. Best PR-AUC (처음 {trials}회 기준): {best_trial.value:.5f}")

        # 3. Reranking (전체 검증셋 기준 재평가)
        if is_val_sampled and (optuna_rerank_delta > 0 or optuna_rerank_ids is not None or interactive_rerank):
            print(f"\n  [Rerank] Reranking 시작 (Full Validation)")
            completed_trials = valid_trials
            
            if completed_trials:
                df_trials = pd.DataFrame([
                    {"Trial": t.number, "Sampled PR-AUC": t.value}
                    for t in completed_trials
                ]).sort_values("Sampled PR-AUC", ascending=False)
                
                print("\n📊 [Optuna Trial 결과 요약]")
                print(df_trials.to_markdown(index=False))
                
                if interactive_rerank:
                    valid_ids = {t.number for t in completed_trials}
                    while True:
                        user_input = input("\n📝 리랭크할 Trial 번호를 쉼표(,)로 구분하여 입력하세요 (Enter 입력 시 자동 선택): ")
                        if not user_input.strip():
                            break
                        try:
                            parsed_ids = [int(x.strip()) for x in user_input.split(",")]
                            invalid_ids = [x for x in parsed_ids if x not in valid_ids]
                            if invalid_ids:
                                print(f"  ⚠️ 에러: 트라이얼 번호 {invalid_ids}는 목록에 없습니다. 다시 입력해주세요.")
                                continue
                            optuna_rerank_ids = parsed_ids
                            break
                        except ValueError:
                            print("  ⚠️ 에러: 숫자와 쉼표(,)만 입력 가능합니다. 다시 입력해주세요.")
                
                if optuna_rerank_ids is not None:
                    candidates = [t for t in completed_trials if t.number in optuna_rerank_ids]
                    print(f"  [Rerank] 지정된 트라이얼 리랭킹 진행: {len(candidates)}개 (IDs: {optuna_rerank_ids})")
                else:
                    best_val = max(t.value for t in completed_trials)
                    candidates = [t for t in completed_trials if t.value >= best_val - optuna_rerank_delta]
                    candidates = sorted(candidates, key=lambda t: t.value, reverse=True)[:optuna_rerank_cap]
                    print(f"  [Rerank] 자동 후보 선택: {len(candidates)}개 (best: {best_val:.5f}, delta: {optuna_rerank_delta})")
                
                best_rerank_score = -1.0
                best_rerank_params = None
                
                for trial_obj in candidates:
                    trial_number = trial_obj.number
                    cand_params = trial_obj.params
                    
                    merged_params = cfg.LGBM_PARAMS.copy()
                    merged_params.update(cand_params)
                    
                    model_dir_name = trial_obj.user_attrs.get("model_dir")
                    trial_dir = optuna_temp_dir / (model_dir_name if model_dir_name else f"trial_{trial_number}")
                    
                    if trial_dir.exists():
                        all_exist = all((trial_dir / f"model_{i}.pkl").exists() for i in range(len(df_train)))
                        if not all_exist:
                            print(f"    - Trial {trial_number} (Skipped): 모델 누락")
                            continue
                        try:
                            print(f"    - Trial {trial_number} 검증 중 (전체 데이터 추론)...", end="\r")
                            models = []
                            for i in range(len(df_train)):
                                model_path = trial_dir / f"model_{i}.pkl"
                                models.append(joblib.load(model_path))
                            
                            X_val = df_val_tune_full[feats]
                            y_val = df_val_tune_full[cfg.TARGET_COL]
                            probs = np.mean([m.predict_proba(X_val)[:, 1] for m in models], axis=0)
                            score = average_precision_score(y_val, probs)
                            print(f"    - Trial {trial_number} (Reuse): Sampled PR-AUC = {trial_obj.value:.5f} -> Full PR-AUC = {score:.5f}")
                            
                            if score > best_rerank_score:
                                best_rerank_score = score
                                best_rerank_params = merged_params
                        except Exception as e:
                            print(f"    - Trial {trial_number} (Skipped): 로드 에러 ({type(e).__name__})")
                            continue
                    else:
                        print(f"    - Trial {trial_number} (Skipped): 저장 모델 없음")
                
                if best_rerank_params is not None:
                    best_params = best_rerank_params
                    print(f"  [Rerank] 최종 선택된 Full PR-AUC: {best_rerank_score:.5f}")
            elif best_trial:
                best_params.update(best_trial.params)
        elif best_trial:
            best_params.update(best_trial.params)
            
    # 4. 최종 학습
    print("\n  [Final] 최적 파라미터로 최종 앙상블 학습 진행...")
    trainer = SubsetTrainer(lgbm_params=best_params, target_col=cfg.TARGET_COL)
    ens     = UnderbaggingEnsemble(trainer=trainer)
    result = ens.fit(df_train, df_val_tune_full, feature_cols=feats, target_col=cfg.TARGET_COL)

    # 임시 디렉토리 제거
    if run_optuna and cleanup_optuna_temp:
        optuna_temp_dir = Path(cfg.MODEL_SAVE_DIR).parent / "optuna_temp"
        if optuna_temp_dir.exists():
            shutil.rmtree(optuna_temp_dir, ignore_errors=True)
            print(f"\n  [Cleanup] 임시 폴더 {optuna_temp_dir} 삭제 완료.")

    return {"ensemble_result": result, "best_params": best_params, "feature_cols": feats}

## 4. 시각화 및 결과 리포트 함수

In [ ]:
def print_ensemble_summary(result: EnsembleResult):
    print("\n" + "="*60)
    print("              UNDERBAGGING ENSEMBLE SUMMARY")
    print("="*60)
    print(f"  Final Ensemble PR-AUC: {result.val_tune_prauc:.5f}")
    print("\n  서브셋별 VAL_TUNE PR-AUC:")
    scores = [r.val_prauc for r in result.subset_results]
    for i, s in enumerate(scores):
        print(f"    Subset {i+1:02d}: {s:.5f}")
    print(f"\n  평균 (단순): {np.mean(scores):.5f}  ±  {np.std(scores):.5f}")
    print("="*60)

def plot_subset_prauc(result: EnsembleResult):
    scores = [r.val_prauc for r in result.subset_results]
    plt.figure(figsize=(10, 4))
    plt.bar(range(1, len(scores)+1), scores, color='skyblue', edgecolor='navy')
    plt.axhline(result.val_tune_prauc, color='red', linestyle='--', label=f'Ensemble ({result.val_tune_prauc:.4f})')
    plt.title('PR-AUC by Subset Model', fontweight='bold')
    plt.xlabel('Subset ID'); plt.ylabel('PR-AUC')
    plt.legend(); plt.grid(axis='y', alpha=0.3)
    plt.show()

def plot_confusion_matrix(result: EnsembleResult, cfg):
    y_true = result.val_tune_y_true
    probs  = result.val_tune_probs
    thresholds = cfg.EVAL_THRESHOLDS
    
    fig, axes = plt.subplots(1, len(thresholds), figsize=(4.5 * len(thresholds), 4.5))
    if len(thresholds) == 1: axes = [axes]

    for ax, thr in zip(axes, thresholds):
        preds = (probs >= thr).astype(int)
        cm    = confusion_matrix(y_true, preds)
        ax.imshow(cm, interpolation='nearest', cmap='Blues')
        
        thresh = cm.max() / 2.
        for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
            count = cm[i, j]
            pct = count / cm.sum() * 100
            ax.text(j, i, f"{count:,}\n({pct:.2f}%)",
                    ha="center", color="white" if count > thresh else "black",
                    fontsize=11, fontweight='bold')

        ax.set_title(f'Threshold = {thr}', fontsize=12, fontweight='bold', pad=15)
        ax.set_xticks([0, 1]); ax.set_xticklabels(['Normal', 'Failure'])
        ax.set_yticks([0, 1]); ax.set_yticklabels(['Normal', 'Failure'])
        ax.set_xlabel('Predicted Label', fontweight='bold')
        ax.set_ylabel('True Label', fontweight='bold')

    plt.suptitle('Confusion Matrix (Count & Percentage)', fontsize=14, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

## 5. 데이터 준비 및 특성 선택

In [ ]:
# 학습용 데이터 10개 서브셋 중 임의의 서브셋 하나를 로드해 피처 수 파악
subset_dir = Path(LocalConfig.SUBSET_DIR)
subset_files = sorted(list(subset_dir.glob("subset_*.parquet")))
if not subset_files:
    raise FileNotFoundError(f"❌ [Error] {subset_dir}에 학습 서브셋 파일이 없습니다.")

df_sample_sub = pd.read_parquet(subset_files[0])
_meta = {'serial_number', 'date', 'failure', 'days_to_failure'}
FEATURE_COLS = [c for c in df_sample_sub.columns if c not in _meta]

df_val_tune = pd.read_parquet(LocalConfig.VAL_TUNE_PATH)
pos_val = df_val_tune[LocalConfig.TARGET_COL].mean()

print(f"학습용 특성 개수 : {len(FEATURE_COLS)} 개")
print(f"검증셋 로우 수    : {len(df_val_tune):,} rows (pos_rate={pos_val:.5f})")

## 6. Optuna 하이퍼파라미터 최적화 & 최종 앙상블 학습

In [ ]:
result_dict = run_training(
    cfg=LocalConfig,
    feature_cols=FEATURE_COLS,
    run_optuna=True,
    optuna_trials=LocalConfig.OPTUNA_TRIALS,
    interactive_rerank=True,
    cleanup_optuna_temp=False,
    optuna_timeout=LocalConfig.OPTUNA_TIMEOUT,
)

ens_result = result_dict['ensemble_result']
best_params = result_dict['best_params']

## 7. 최종 평가 리포트

In [ ]:
print('\n=========================================')
print('🏆 최종 선택된 최적 하이퍼파라미터')
print('=========================================')
for k, v in best_params.items():
    print(f'  - {k}: {v}')

print_ensemble_summary(ens_result)
plot_subset_prauc(ens_result)
plot_confusion_matrix(ens_result, LocalConfig)

## 8. 모델 및 학습 설정 저장

In [ ]:
SAVE_DIR = Path(LocalConfig.MODEL_SAVE_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 10개 서브셋 모델 pkl 저장
for i, model in enumerate(ens_result.models):
    path = SAVE_DIR / f'subset_{i:02d}.pkl'
    joblib.dump(model, path)
    print(f'  [Saved] 모델 저장: {path}')

# 학습 특성 정의 json 저장
feat_path = SAVE_DIR / 'feature_cols.json'
with open(feat_path, 'w', encoding='utf-8') as f:
    json.dump(result_dict['feature_cols'], f, ensure_ascii=False, indent=2)
print(f'  [Saved] 피처 목록: {feat_path}')

# best_params json 저장
param_path = SAVE_DIR / 'best_params.json'
with open(param_path, 'w', encoding='utf-8') as f:
    json.dump(best_params, f, ensure_ascii=False, indent=2)
print(f'  [Saved] 최적 파라미터: {param_path}')

## 9. 학습 결과물 및 파라미터 파일 무결성 검증 테스트 (Verification Tests)

최종적으로 저장된 10개의 서브셋 모델 파일(`subset_00.pkl` ~ `subset_09.pkl`), 피처 컬럼 정의 파일(`feature_cols.json`), 그리고 최적화된 하이퍼파라미터 설정 파일(`best_params.json`)의 저장 여부 및 정상 로딩 상태를 검증합니다.

In [ ]:
SAVE_DIR = Path(LocalConfig.MODEL_SAVE_DIR)

print("🔍 [6-D단계 무결성 검증] 시작...")

try:
    # 1. 10개 서브셋 모델 pkl 파일 물리 저장 확인 및 로딩 테스트
    print("Test 1: 10개 서브셋 모델 저장 및 로딩 검증")
    for i in range(10):
        path = SAVE_DIR / f'subset_{i:02d}.pkl'
        assert path.is_file(), f"오류: 모델 파일 {path.name}이 존재하지 않습니다."
        model = joblib.load(path)
        # LightGBM Classifier 클래스인지 검증
        assert hasattr(model, 'predict_proba'), f"오류: {path.name} 파일은 올바른 모델 객체가 아닙니다."
    print("  -> [PASS] 10개 서브셋 모델 pkl 파일 무결성 확인 완료.")

    # 2. 피처 컬럼 json 파일 검증
    print("Test 2: feature_cols.json 저장 및 로딩 검증")
    feat_path = SAVE_DIR / 'feature_cols.json'
    assert feat_path.is_file(), "오류: feature_cols.json 파일이 존재하지 않습니다."
    with open(feat_path, 'r', encoding='utf-8') as f:
        feats = json.load(f)
    assert isinstance(feats, list) and len(feats) > 0, "오류: feature_cols.json 형식이 올바르지 않습니다."
    print(f"  -> [PASS] feature_cols.json 무결성 확인 완료. (총 {len(feats)}개 피처)")

    # 3. 최적 파라미터 json 파일 검증
    print("Test 3: best_params.json 저장 및 로딩 검증")
    param_path = SAVE_DIR / 'best_params.json'
    assert param_path.is_file(), "오류: best_params.json 파일이 존재하지 않습니다."
    with open(param_path, 'r', encoding='utf-8') as f:
        params = json.load(f)
    assert isinstance(params, dict) and 'n_estimators' in params, "오류: best_params.json 형식이 올바르지 않거나 필수 키가 누락되었습니다."
    print("  -> [PASS] best_params.json 무결성 확인 완료.")

    print("\n🏆 [6-D단계 정합성 검증 완료] 모든 학습 결과물이 정상적으로 저장되고 로드 가능합니다!")
finally:
    pass